# Notebook 2: LSTM Forecasting and XGBoost Refinement

Each IMF component is modelled using LSTM and refined using XGBoost.

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from xgboost import XGBRegressor


In [ ]:
# Convert IMF series into supervised learning format
def create_dataset(series, window=30):
    X=[]; y=[]
    for i in range(len(series)-window):
        X.append(series[i:i+window])
        y.append(series[i+window])
    return np.array(X), np.array(y)


In [ ]:
# LSTM architecture
def build_lstm(units=64, dropout=0.2, lr=0.001):
    model=Sequential([
        LSTM(units,input_shape=(30,1)),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr),loss='mse')
    return model


In [ ]:
# Train LSTM for every IMF
imf_predictions=[]
for i, imf in enumerate(imfs):
    X,y=create_dataset(imf)
    X=X.reshape(X.shape[0],X.shape[1],1)
    model=build_lstm()
    model.fit(X,y,epochs=50,batch_size=32,verbose=0)
    pred=model.predict(X,verbose=0).flatten()
    imf_predictions.append(pred)


In [ ]:
# XGBoost residual refinement
xgb_predictions=[]
for pred in imf_predictions:
    X=np.arange(len(pred)).reshape(-1,1)
    xgb=XGBRegressor(n_estimators=200,max_depth=5,learning_rate=0.05)
    xgb.fit(X,pred)
    xgb_predictions.append(xgb.predict(X))

final_prediction=np.sum(xgb_predictions,axis=0)
